# Gold: star schema, published only if the checks pass

Builds the dimensions and facts into `stg_*` tables, audits them, and publishes the gold tables the Direct Lake model reads only when every ERROR check passes.

Requires the `env_portfolio` Environment (with the `portfolio_migration` wheel) and the `lh_portfolio` lakehouse attached as default. Logic and tests live in the GitHub repo; do not edit logic here.

In [ ]:
# Set by the data pipeline to @pipeline().RunId. Empty means a manual run.
batch_id = ""

In [ ]:
import json
import time
import uuid
from dataclasses import asdict

import portfolio_migration
from portfolio_migration.lakehouse.io import FabricLake

batch_id = batch_id or f"manual-{uuid.uuid4()}"
print(f"portfolio_migration {portfolio_migration.__version__}, batch {batch_id}")
lake = FabricLake(spark)
started = time.perf_counter()

In [ ]:
from portfolio_migration.lakehouse import gold

LAYER = "gold"
# Writes the star schema to stg_* tables, runs the gold checks against those, and
# publishes the gold tables only if no ERROR check fails.
result = gold.run(lake, batch_id)
output = [asdict(r) for r in result.tables]
display(spark.createDataFrame(output))
display(spark.createDataFrame([asdict(c) for c in result.checks]))
print(f"{len(result.checks)} gold checks passed before publishing")

In [ ]:
seconds = round(time.perf_counter() - started, 1)
print(f"finished in {seconds} s")
notebookutils.notebook.exit(json.dumps({"layer": LAYER, "batch_id": batch_id, "seconds": seconds,
                                        "output": output}))